In [1]:
#importing

import cv2
import numpy as np
import tensorflow as tf
import time

#load Model
path_to_model = r"D:\Cruz\AI Project\Image Dedection\Model\ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8\saved_model"

print('loading_model')

detect_fn = tf.saved_model.load(path_to_model)

print('Model_Load')

# open webcam

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)


cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

#variables
prev_frame_time = 0
frame_count = 0
fps = 0

#store previous detection
last_boxes = []
last_classes = []
last_scores = []

#class names

class_names = {1: 'person',2: 'bicycle',3: 'car',4: 'motorbike',5: 'aeroplane',
            6: 'bus',7: 'train',8: 'truck',9: 'boat',10: 'traffic light',11: 'fire hydrant',13: 'stop sign',
            14: 'parking meter',15: 'bench',16: 'bird',17: 'cat',18: 'dog',19: 'horse',20: 'sheep',21: 'cow',
            22: 'elephant',23: 'bear', 24: 'zebra',25: 'giraffe',27: 'backpack',28: 'umbrella',31: 'handbag',
            32: 'tie',33: 'suitcase',34: 'frisbee',35: 'skis',36: 'snowboard',37: 'sports ball',38: 'kite',
            39: 'baseball bat',40: 'baseball glove',41: 'skateboard',42: 'surfboard',43: 'tennis racket',
            44: 'bottle',46: 'wine glass',47: 'cup',48: 'fork',49: 'knife',50: 'spoon',51: 'bowl',52: 'banana',
            53: 'apple',54: 'sandwich',55: 'orange',56: 'broccoli',57: 'carrot',58: 'hot dog',59: 'pizza',
            60: 'donut',61: 'cake',62: 'chair',63: 'sofa',64: 'potted plant',65: 'bed',67: 'dining table',
            70: 'toilet',72: 'tv',73: 'laptop',74: 'mouse',75: 'remote',76: 'keyboard',77: 'cell phone',
            78: 'microwave',79: 'oven',80: 'toaster',81: 'sink',82: 'refrigerator',84: 'book',85: 'clock',
            86: 'vase',87: 'scissors',88: 'teddy bear',89: 'hair drier',90: 'toothbrush'}

#main loop

while True:
    
    ret, frame = cap.read()
    
    if not ret:
       break
    
    #frame counter
    frame_count += 1
    height, width, _ = frame.shape

    # RUN DETECTION EVERY 3RD FRAME

    if frame_count % 3 == 0:

        #convert to tensor

        input_tensor = tf.convert_to_tensor(frame)
        input_tensor = input_tensor[tf.newaxis, ...]
        
        #detection 

        detections = detect_fn(input_tensor)

        #get output

        last_boxes = detections['detection_boxes'][0].numpy()

        last_classes = detections['detection_classes'][0].numpy().astype(int)

        last_scores = detections['detection_scores'][0].numpy()


        
    for i in range(min(5,len(last_scores))):
        

        if last_scores[i] > 0.4:
            ymin, xmin, ymax, xmax = last_boxes[i]

            left = int(xmin * width)
            right = int(xmax * width)

            top = int(ymin * height)
            bottom = int(ymax * height)

            #draw rectangle
            cv2.rectangle(frame,(left,top),(right,bottom),(0,255,0),2)

            #label

            class_id = last_classes[i]

            object_name = class_names.get(class_id, "Unknown")

            label = f"{object_name}: {last_scores[i]:.2f}"

            cv2.putText(frame,label,(left,top-10),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),2)


    new_frame_time = time.time()

    fps = 0.9 * fps + 0.1 * (1 / max(new_frame_time - prev_frame_time, 0.0001))

    prev_frame_time = new_frame_time

    cv2.putText(
            frame,
            f"FPS: {int(fps)}",
            (20,40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255,0,0),
                2
)        

    cv2.imshow("Object Detection", frame)

    #press q to exit and #press s to capture

    key = cv2.waitKey(1) & 0xFF 
    if key == ord('q'):
        break
    
    if key == ord('s'):
    # Creates a unique name like "capture_1683400000.jpg"
        filename = f"capture_{int(time.time())}.jpg" 
        cv2.imwrite(filename, frame)
        print(f'Saved as {filename}')

cap.release()
cv2.destroyAllWindows()
        



loading_model
Model_Load


KeyboardInterrupt: 